# Housing Unit Allocation with Person Record File Full Workflow

## Overview
This code works with the National Structures Inventory to run the housing unit allocation (HUA) and the person record file (PREC) workflow.
The HUA process is generalizable to any county in the United States. The HUA process will work for any file that has locations of structures and some basic information about the buildings.
The process is designed to work with [IN-CORE](https://incore.ncsa.illinois.edu/), a community resilience modeling environment.
Using IN-CORE requires an account and access to the IN-CORE Dataservice.

Functions are provided to obtain and clean data required for the version 2 Housing Unit Allocation. 

## Required Inputs
Program requires the following inputs:
If using the National Structures Inventory there are no required inputs.
    
## Output Description
The output of this workflow is a CSV file with the housing unit inventory allocated to a building inventory using the housing unit allocation model.

The output CSV is designed to be used in the Interdependent Networked Community Resilience Modeling Environment (IN-CORE).

IN-CORE is an open source python package that can be used to model the resilience of a community. To download IN-CORE, see:

https://incore.ncsa.illinois.edu/


## Instructions
Users can run the workflow by executing each block of code in the notebook.

## Description of Program
- program:    ncoda_07fv1_HUA_PREC_NSI
- task:       Start with NSI building inventory, run housing unit allocation algorithm, and then run person record file algorithm
- See github commits for description of program updates
- Current Version: v1 - 
- 2024-02-20 - Combine code from 07c, 07d, and 07e into one notebook
- 2024-05-22 - removed the drop down menu, did not work consistently
- project:    Interdependent Networked Community Resilience Modeling Environment (IN-CORE), Subtask 5.2 - Social Institutions
- funding:	  NIST Financial Assistance Award Numbers: 70NANB15H044 and 70NANB20H008 
- author:     Nathanael Rosenheim

## Required Citations:
Rosenheim, Nathanael, Roberto Guidotti, Paolo Gardoni & Walter Gillis Peacock. (2021). Integration of detailed household and housing unit characteristic data with critical infrastructure for post-hazard resilience modeling. _Sustainable and Resilient Infrastructure_. 6(6), 385-401. https://doi.org/10.1080/23789689.2019.1681821

Rosenheim, Nathanael (2021) “Detailed Household and Housing Unit Characteristics: Data and Replication Code.” _DesignSafe-CI_. 
https://doi.org/10.17603/ds2-jwf6-s535.

In [1]:
# To reload submodules need to use this magic command to set autoreload on
%load_ext autoreload
%autoreload 2
import pyncoda.ncoda_00g_community_options as community_options 
from IPython.display import display

### How to set up the Community Dictionary
Please review the python code in the file pyncoda/ncoda_00g_community_options.py

In this file you will find a collection of data dictionaries with various ways to setup the inputs for the Housing Unit Allocation process. 

The basic dictionary includes the name of the community, the county FIPS code, your input building inventory file, and key variables in the building inventory file.

In [2]:
# select a community from this list
# if your community is not in this list, add it to the file ncoda_00g_community_options.py
community_options.list_community_options(community_options.communities_dictionary)

['Lumberton, NC: IN-CORE Building inventory for Robeson County, NC',
 'Galveston, TX: IN-CORE Building inventory for Galveston County, TX',
 'Galveston, TX: NSI Building inventory for Galveston County, TX',
 'Galveston, TX: IN-CORE Building inventory for Galveston Island, TX',
 'Mayfield, KY: NSI Building inventory for Graves County, KY',
 'Beaumont, TX: NSI Building inventory for Jefferson County, TX',
 'Beaumont, TX: Safayet Building inventory for Jefferson County, TX',
 'Pentwater, MI: NSI Building inventory for Oceana County, MI',
 'Seaside, OR: NSI Building inventory for Clatsop County, OR',
 'Lane County, OR: NSI Building inventory for Lane County, OR',
 'Benton County, OR: NSI Building inventory for Benton County, OR',
 'Southeast Texas Urban Integrated Field Lab: NSI Building inventory for Southeast Texas',
 'Brazos County, TX: NSI Building inventory for Brazos County, TX']

In [3]:
community_id_by_name = 'Brazos County, TX: NSI Building inventory for Brazos County, TX'

In [4]:
community_id, focalplace, countyname, countyfips = community_options.get_community_id_by_name(community_id_by_name)
communities = {community_id : community_options.communities_dictionary[community_id]}

Selected community ID: Brazos_TX_NSI
Brazos County, TX is in TEXAS
Focal place: Bryan
Brazos County, TX is in Brazos County, TX with FIPS code 48041
Use IN-CORE: False


In [5]:
communities

{'Brazos_TX_NSI': {'community_name': 'Brazos County, TX',
  'focalplace_name': 'Bryan',
  'STATE': 'TEXAS',
  'years': ['2010'],
  'counties': {1: {'FIPS Code': '48041', 'Name': 'Brazos County, TX'}},
  'building_inventory': {'use_incore': False,
   'id': 'NSI',
   'note': 'NSI Building inventory for Brazos County, TX',
   'archetype_var': 'occtype',
   'bldg_uniqueid': 'fd_id_bid',
   'residential_archetypes': {'RES1-1SNB': {'Description': 'Single Family Dwelling',
     'HU estimate': 1},
    'RES1-1SWB': {'Description': 'Single Family Dwelling', 'HU estimate': 1},
    'RES1-2SNB': {'Description': 'Single Family Dwelling', 'HU estimate': 1},
    'RES1-2SWB': {'Description': 'Single Family Dwelling', 'HU estimate': 1},
    'RES1-3SNB': {'Description': 'Single Family Dwelling', 'HU estimate': 1},
    'RES1-3SWB': {'Description': 'Single Family Dwelling', 'HU estimate': 1},
    'RES1-SLNB': {'Description': 'Single Family Dwelling', 'HU estimate': 1},
    'RES1-SLWB': {'Description': 'Sin

## Setup Python Environment

In [6]:
import pandas as pd
import geopandas as gpd # For reading in shapefiles
import numpy as np
import sys # For displaying package versions
import os # For managing directories and file paths if drive is mounted
import scooby # Reports Python environment

import contextily as cx # For adding basemap tiles to plot
import matplotlib.pyplot as plt # For plotting and making graphs

In [7]:
# open, read, and execute python program with reusable commands
import pyncoda.ncoda_00d_cleanvarsutils as cleanvarsutils
import pyncoda.ncoda_04c_poptableresults as poptableresults
from pyncoda.ncoda_07i_process_communities import process_community_workflow

In [8]:
# Generate report of Python environment
base_packages = ['pandas','ipyleaflet','seaborn','contextily']
incore_packages = ['pyincore','pyincore_viz']
check_packages = base_packages + incore_packages
print(scooby.Report(additional=check_packages))


--------------------------------------------------------------------------------
  Date: Tue Dec 10 14:54:19 2024 PST

                OS : Darwin (macOS 15.1.1)
            CPU(s) : 10
           Machine : arm64
      Architecture : 64bit
               RAM : 32.0 GiB
       Environment : Jupyter
       File system : apfs

  Python 3.12.8 (v3.12.8:2dc476bcb91, Dec  3 2024, 14:43:19) [Clang 13.0.0
  (clang-1300.0.29.30)]

            pandas : 2.2.2
        ipyleaflet : Module not found
           seaborn : 0.13.2
        contextily : 1.6.0
          pyincore : Module not found
      pyincore_viz : Module not found
             numpy : 1.26.4
             scipy : 1.14.1
           IPython : 8.30.0
        matplotlib : 3.8.4
            scooby : 0.10.0
--------------------------------------------------------------------------------


In [9]:
# Check working directory - good practice for relative path access
os.getcwd()

'/Users/adamzs/Repos/intersect-community-data'

## Run Housing Unit Allocation
The following code will produce the following outputs:
1. Housing Unit Inventory
2. Address Point Inventory
3. Housing Unit Allocation

In [10]:
workflow = process_community_workflow(communities, seed=9876)

### Process Communities

```
hua_hui_gdf = workflow.process_communities()
```

in `ncode_07i_process_communities.py`

#### Prep

In [11]:
from pyncoda.ncoda_00b_directory_design import directory_design # To initialize the output folder(s)

# assuming a specific community is requested
community = list(workflow.communities.keys())[0]

outputfolders = directory_design(
    state_county_name = community,
    outputfolder = workflow.outputfolder)

# community dictionary
community_dict = workflow.communities[community]
use_incore = community_dict['building_inventory']['use_incore']

In [17]:
# Load Housing Unit Inventory
housing_unit_inv_df, hui_dataset_id = \
    workflow.generate_hui(workflow.communities, use_incore)

Generating Housing Unit Inventory v2.0.0 data for Brazos County, TX
Brazos County, TX : county FIPS Code 48041
File already exists, skipping: OutputData/Brazos_TX_NSI/../hui_v2-0-0_Brazos_TX_NSI_2010_rs9876.csv
Checking output for huid
Checking output for blockid
Checking output for bgid
Checking output for tractid
Checking output for FIPScounty
Checking output for numprec
Checking output for ownershp
Checking output for race
Checking output for hispan
Checking output for family
Checking output for vacancy
Checking output for gqtype
Checking output for incomegroup
Checking output for hhinc
Checking output for randincome
Checking output for poverty
Checking huid Data Type
   Current type: <class 'pandas.core.series.Series'> Expected type <class 'str'>
Checking blockid Data Type
   Current type: <class 'pandas.core.series.Series'> Expected type <class 'str'>
    Length of blockid is correct
Checking bgid Data Type
   Current type: <class 'pandas.core.series.Series'> Expected type <class 

/Users/adamzs/Repos/intersect-community-data/pyncoda/ncoda_07a_generate_hui.py:152: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  hui_incore_df_fixed = hui_incore_df.applymap(lambda cell: int(cell) if str(cell).endswith('.0') else cell)


#### Load Housing Unit Inventory

```
housing_unit_inv_df, hui_dataset_id = \
                self.generate_hui(self.communities, use_incore)
```

in `ncoda_07i_process_communities`

In [13]:
from pyncoda.ncoda_07a_generate_hui import generate_hui_functions # to load Housing Unit Inventory

# this is not really a dataframe, rather an object
generate_hui_df = generate_hui_functions(
    communities =   communities,
    seed =          workflow.seed,
    version =       workflow.version,
    version_text=   workflow.version_text,
    basevintage=    workflow.basevintage,
    outputfolder=   workflow.outputfolder,
    use_incore=     use_incore
)

hui_dataset_id = generate_hui_df.generate_hui_v2_for_incore()

housing_unit_inv_df = hui_dataset_id
hui_dataset_id = 'local'

housing_unit_inv_df, hui_dataset_id = (housing_unit_inv_df, hui_dataset_id)

Generating Housing Unit Inventory v2.0.0 data for Brazos County, TX
Brazos County, TX : county FIPS Code 48041
File already exists, skipping: OutputData/Brazos_TX_NSI/../hui_v2-0-0_Brazos_TX_NSI_2010_rs9876.csv
Checking output for huid
Checking output for blockid
Checking output for bgid
Checking output for tractid
Checking output for FIPScounty
Checking output for numprec
Checking output for ownershp
Checking output for race
Checking output for hispan
Checking output for family
Checking output for vacancy
Checking output for gqtype
Checking output for incomegroup
Checking output for hhinc
Checking output for randincome
Checking output for poverty
Checking huid Data Type
   Current type: <class 'pandas.core.series.Series'> Expected type <class 'str'>
Checking blockid Data Type
   Current type: <class 'pandas.core.series.Series'> Expected type <class 'str'>
    Length of blockid is correct
Checking bgid Data Type
   Current type: <class 'pandas.core.series.Series'> Expected type <class 

/Users/adamzs/Repos/intersect-community-data/pyncoda/ncoda_07a_generate_hui.py:152: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  hui_incore_df_fixed = hui_incore_df.applymap(lambda cell: int(cell) if str(cell).endswith('.0') else cell)


##### Generate HUI data for IN-CORE

```
hui_dataset_id = generate_hui_df.generate_hui_v2_for_incore()
```

in `ncoda_07a_generate_hui`

In [56]:
# Create empty container to store outputs for in-core
# Will use these to combine multiple counties
hui_incore_county_df = {}
title = "Housing Unit Inventory v2.0.0 data for "+communities[community]['community_name']
print("Generating",title)
output_filename = f'hui_{generate_hui_df.version_text}_{community}_{generate_hui_df.basevintage}_rs{generate_hui_df.seed}'

county_list = ''

# Workflow for generating HUI data for IN-CORE
county = [county for county in communities[community]['counties'].keys()][0]
state_county = communities[community]['counties'][county]['FIPS Code']
state_county_name  = communities[community]['counties'][county]['Name']
print(state_county_name,': county FIPS Code',state_county)
county_list = county_list + state_county_name+': county FIPS Code '+state_county

# create output folders for hui data generation - this seems redundant
outputfolders = directory_design(state_county_name = community,
                                 outputfolder = generate_hui_df.outputfolder)

from pyncoda.CommunitySourceData.api_census_gov.acg_05a_hui_functions import hui_workflow_functions

generate_df = hui_workflow_functions(
    state_county = state_county,
    state_county_name= state_county_name,
    seed = generate_hui_df.seed,
    version = generate_hui_df.version,
    version_text = generate_hui_df.version_text,
    basevintage = generate_hui_df.basevintage,
    outputfolder = generate_hui_df.outputfolder,
    outputfolders = outputfolders)

# Check if output file already exists - no need for this now
"""
check_file = outputfolders['top']+"/../"+output_filename+'.csv'
if os.path.exists(check_file):
    print("File already exists, skipping:",check_file)
    # Read in HUI Data
    hui_df = pd.read_csv(check_file, header="infer")
    # Save version for IN-CORE in v2 format
    hui_incore_df = \
        generate_df.save_incore_version2(hui_df)
    # Remove .0 from data
    hui_incore_df_fixed = hui_incore_df.applymap(lambda cell: int(cell) if str(cell).endswith('.0') else cell)

    return hui_incore_df_fixed
"""

# Generate base housing unit inventory
# base_hui_df = generate_df.run_hui_workflow()

Generating Housing Unit Inventory v2.0.0 data for Brazos County, TX
Brazos County, TX : county FIPS Code 48041


'\ncheck_file = outputfolders[\'top\']+"/../"+output_filename+\'.csv\'\nif os.path.exists(check_file):\n    print("File already exists, skipping:",check_file)\n    # Read in HUI Data\n    hui_df = pd.read_csv(check_file, header="infer")\n    # Save version for IN-CORE in v2 format\n    hui_incore_df =         generate_df.save_incore_version2(hui_df)\n    # Remove .0 from data\n    hui_incore_df_fixed = hui_incore_df.applymap(lambda cell: int(cell) if str(cell).endswith(\'.0\') else cell)\n\n    return hui_incore_df_fixed\n'

###### Generate base housing unit inventory

```
base_hui_df = generate_df.run_hui_workflow()
```

in `acg_05a_hui_functions.py`

In [57]:
# This is where the money is - let's break it up / in acg_05a_hui_functions.py

savelog = False

"""
Workflow to produce Housing Unit Inventory
"""
# Start empty containers to store block level and tract level data
tract_df = {}
block_df = {}

# Save output description as text
output_filename = f'hui_{generate_df.version_text}_{generate_df.state_county}_{generate_df.basevintage}_rs{generate_df.seed}'
generate_df.output_filename = output_filename

# Save output as a log file function
from pyncoda import ncoda_00c_save_output_log as logfile

if savelog == True:
    log_filepath = generate_df.outputfolders['logfiles']+"/"+output_filename+'.log'
    # start log file
    logfile.start(log_filepath)
    #generate_df.save_environment_version_details()
    
print("\n***************************************")
print("    Obtain and clean core housing unit characteristics for",generate_df.state_county_name)
print("***************************************\n")

from pyncoda.CommunitySourceData.api_census_gov.acg_01a_BaseInventory import BaseInventory

print(generate_df.outputfolders)
block_df['core'] = BaseInventory.get_apidata(state_county = generate_df.state_county,
                                    outputfolders = generate_df.outputfolders,
                                    outputfile = "CoreHUI")


***************************************
    Obtain and clean core housing unit characteristics for Brazos County, TX
***************************************

{'top': 'OutputData/Brazos_TX_NSI', 'logfiles': 'OutputData/Brazos_TX_NSI/00_logfiles', 'CommunitySourceData': 'OutputData/Brazos_TX_NSI/01_CommunitySourceData', 'TidyCommunitySourceData': 'OutputData/Brazos_TX_NSI/02_TidyCommunitySourceData', 'BaseInventory': 'OutputData/Brazos_TX_NSI/03_BaseInventory', 'RandomMerge': 'OutputData/Brazos_TX_NSI/04_RandomMerge', 'Verify': 'OutputData/Brazos_TX_NSI/05_Verify', 'Explore': 'OutputData/Brazos_TX_NSI/06_Explore', 'Uncertainty_propagation': 'OutputData/Brazos_TX_NSI/07_Uncertainty_propagation', 'Validation': 'OutputData/Brazos_TX_NSI/08_Validation'}
File OutputData/Brazos_TX_NSI/03_BaseInventory/CoreHUI_48041_2010.csv Already exists - Skipping API Call.


In [58]:
from pyncoda.CommunitySourceData.api_census_gov.acg_00b_hui_block2010 import family_byrace_P18_varstem_roots

# Use specific tables from Census to get family count and update family attribute with it
block_df['family'] = BaseInventory.graft_on_new_char(base_inventory= block_df['core'],
                            state_county = generate_df.state_county,
                            new_char = 'family',
                            new_char_dictionaries = [family_byrace_P18_varstem_roots],
                            outputfile = "hui",
                            outputfolders = generate_df.outputfolders)

#display(block_df['family'])

#display(block_df['family']['hispan'].value_counts())

File OutputData/Brazos_TX_NSI/03_BaseInventory/hui_family_48041_2010.csv Already exists with new variable grafted - Skipping API Call.


In [59]:
from pyncoda.CommunitySourceData.api_census_gov.acg_00c_hispan_block2010 import (
    tenure_size_H16HAI_varstem_roots,
    hispan_byrace_H7_varstem_roots,
    tenure_byhispan_H15_varstem_roots
)

# Use tables from census to get hispanic count and update hispanic attribute with it
block_df['hispan'] = BaseInventory.graft_on_new_char(base_inventory= block_df['family'],
                                    state_county = generate_df.state_county,
                                    new_char = 'hispan',
                                    new_char_dictionaries = 
                                    [tenure_size_H16HAI_varstem_roots,
                                        hispan_byrace_H7_varstem_roots,
                                        tenure_byhispan_H15_varstem_roots
                                        ],
                                    basevintage = "2010", 
                                    basegeolevel = 'Block',
                                    outputfile = "hui",
                                    outputfolders = generate_df.outputfolders)

File OutputData/Brazos_TX_NSI/03_BaseInventory/hui_hispan_48041_2010.csv Already exists with new variable grafted - Skipping API Call.


In [60]:
from pyncoda.CommunitySourceData.api_census_gov.acg_00d_hhinc_ACS5yr2012 import (
    hhinc_varstem_roots,
    family_varstem_roots
)

# Use tables from ACS to get household income for households by census tract
tract_df["B19001"] = BaseInventory.get_apidata(state_county = generate_df.state_county,
                                    geo_level = 'tract',
                                    vintage = "2012", 
                                    mutually_exclusive_varstems_roots_dictionaries =
                                                        [hhinc_varstem_roots],
                                    outputfolders = generate_df.outputfolders,
                                    outputfile = "B19001")

# Use tables from ACS to get family income for households by census tract
tract_df["B19101"] = BaseInventory.get_apidata(state_county = generate_df.state_county,
                                    geo_level = 'tract',
                                    vintage = "2012", 
                                    mutually_exclusive_varstems_roots_dictionaries =
                                                        [family_varstem_roots],
                                    outputfolders = generate_df.outputfolders,
                                    outputfile = "B19101")

File OutputData/Brazos_TX_NSI/03_BaseInventory/B19001_48041_2012.csv Already exists - Skipping API Call.
File OutputData/Brazos_TX_NSI/03_BaseInventory/B19101_48041_2012.csv Already exists - Skipping API Call.


In [61]:
from pyncoda.CommunitySourceData.api_census_gov.acg_02a_add_categorical_char \
     import add_new_char_by_random_merge_2dfs

# Now, merge the tract-level income information for households and families

# Note that the add_new_char... is not a function, but a class!
# The call below just initializes the object
income_by_family = add_new_char_by_random_merge_2dfs(
    dfs = {
        'primary'  : {
            'data': tract_df['B19001'], 
            'primarykey' : 'uniqueidB19001',
            'geolevel' : 'Tract',
            'geovintage' :'2010',
            'notes' : 'Household level data with income.'
        },
        'secondary' : {
            'data': tract_df['B19101'], 
            'primarykey' : 'uniqueidB19101',
            'geolevel' : 'Tract',
            'geovintage' :'2010',
            'notes' : 'Family level data with income.'
        }
    },
    seed = generate_df.seed,
    common_group_vars = ['incomegroup'],
    new_char = 'family',
    geolevel = "Tract",
    geovintage = "2010",
    by_groups = {'All' : {'by_variables' : ['race','hispan']}},
    fillna_value = 0,
    state_county = generate_df.state_county,
    outputfile = "B19001rmB19101",
    outputfolder = generate_df.outputfolders['RandomMerge'],
    savefiles = generate_df.savefiles
)

import json

# Then, this function returns a hard-coded dictionary with some dynamic elements
rounds = income_by_family.make_round_options_dict()
#print(json.dumps(rounds, indent=2))

# And this is where the actual merge happens
tract_income_match = income_by_family.run_random_merge_2dfs(rounds)

Round 1
Performing random merge at geography level: Tract
By original common group vars and by groups variables.
Running random merge by ['Tract2010', 'race', 'hispan', 'incomegroup']
    Setting up  primary data with primary key and flags
Longest tract : 900
tract Expected Length 6 Available Length 3
Geolevels available ['state', 'county']
Geolvarids available []
Adding Tract2010 expected length 11
Dataframe has GEO_ID for new geovar Tract2010
Confirming Tract2010 has expected length.
Checking primary key name uniqueidB19001
Primary key variable uniqueidB19001 is unique.
Primary key uniqueidB19001 has no missing values
Initializing primary flag set variable for family
New flag: family_flagsetrm
['uniqueidB19001', 'Tract2010', 'index', 'GEO_ID', 'state', 'county', 'tract', 'race', 'hispan', 'incomegroup', 'hu_counter']
Initializing geovar flag set variable for family at geolevel Tract2010
New flag: family_Tract2010_flagsetrm
Observations without primary flag set 72197
Observations with

/Users/adamzs/Repos/intersect-community-data/pyncoda/CommunitySourceData/api_census_gov/acg_02a_add_categorical_char.py:763: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.5' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  set_flag_df.loc[conditions, self.flaggeo_var] = .5


After update observations geovar flag set 3 = 0
    Check random merge results for primary data.
Check by geovar flag family_County2010_flagsetrm
Observations flag not equal to 0 40829
Input and output data have the same length 72197
Outputdata has 40829 Observations with predicted family
Percent left to predict: 43.45
    Overwrite input data with update output data.
Flag vars available ['family_flagsetrm', 'family_Tract2010_flagsetrm', 'family_County2010_flagsetrm']
Before update output_df observations flag set 3 = 0
Before update results observations flag set 3 = 0
After update observations geovar flag set 3 = 0
    Check random merge results for secondary data.
Check by geovar flag family_County2010_flagsetrm
Observations flag not equal to 0 40829
Input and output data have the same length 40829
Outputdata has 40829 Observations with predicted family
Percent left to predict:  0.00
    Overwrite input data with update output data.

+++++++++++++++++++++++++++++++++++++++
Percent lef

In [62]:
# And then merge the income data with the block-based household data

block_income = add_new_char_by_random_merge_2dfs(
    dfs = {'primary'  : {'data': block_df['hispan'], 
                    'primarykey' : 'huid',
                    'geolevel' : 'Block',
                    'geovintage' :'2010',
                    'notes' : 'Household level data without income.'},
        'secondary' : {'data': tract_income_match['primary'], 
                    'primarykey' : 'uniqueidB19001',
                    'geolevel' : 'Tract',
                    'geovintage' :'2010',
                    'notes' : 'Household and Family level data with income.'}},
    seed = generate_df.seed,
    common_group_vars = ['family'],
    new_char = 'incomegroup',
    geolevel = "Tract",
    geovintage = "2010",
    by_groups = {'Hispanic'     : {'by_variables' : ['hispan']},
                'not Hispanic' : {'by_variables' : ['race']}},
    fillna_value= -999,
    state_county = generate_df.state_county,
    outputfile = "hui_B19001rmB19101",
    outputfolder = generate_df.outputfolders['RandomMerge'],
    reuse_secondary = True,
    savefiles = generate_df.savefiles
)

# Set up round options
rounds = block_income.make_round_options_dict()

block_income_df = block_income.run_random_merge_2dfs(rounds)

File OutputData/Brazos_TX_NSI/04_RandomMerge/hui_B19001rmB19101_48041_2010_rs9876_primary.csv Already exists - Skipping Random Merge.
File hui_B19001rmB19101_48041_2010_rs9876_secondary Already exists - Skipping Random Merge.


In [64]:
# Save output as a log file function
# Stop log file
if savelog == True:
    logfile.stop()

base_hui_df = block_income_df

###### Polish and save result

In [65]:
hui_df = generate_df.final_polish_hui(base_hui_df['primary'])


***************************************
    Try to polish final hui data.
***************************************

Add random income.
Add poverty.
Make category 0 for numprec, vacancy and gqtype
Drop extra columns.

***************************************
    Save cleaned data file.
***************************************

File saved: /Users/adamzs/Repos/intersect-community-data/OutputData/Brazos_TX_NSI/hui_v2-0-0_48041_2010_rs9876.csv


In [66]:
# Save version for IN-CORE in v2 format
hui_incore_county_df[state_county] = \
    generate_df.save_incore_version2(hui_df)

Checking output for huid
Checking output for blockid
   Attempt to add blockid
   Check current version for Block2010
    Renaming Block2010 blockid
Checking output for bgid
   Attempt to add bgid
   Check current version for BlockGroup2010
    Attempt to make bgid
Checking output for tractid
   Attempt to add tractid
   Check current version for Tract2010
    Renaming Tract2010 tractid
Checking output for FIPScounty
   Attempt to add FIPScounty
   Check current version for County2010
    Renaming County2010 FIPScounty
Checking output for numprec
Checking output for ownershp
Checking output for race
Checking output for hispan
Checking output for family
Checking output for vacancy
Checking output for gqtype
Checking output for incomegroup
Checking output for hhinc
Checking output for randincome
   Attempt to add randincome
   Check current version for randincomeB19101
    Renaming randincomeB19101 randincome
Checking output for poverty
Checking huid Data Type
   Current type: <class 'pa

In [67]:
# combine multiple counties
hui_incore_df = pd.concat(hui_incore_county_df.values(), 
                                ignore_index=True, axis=0)

# Remove .0 from data
hui_incore_df_fixed = hui_incore_df.applymap(lambda cell: int(cell) if str(cell).endswith('.0') else cell)

#Save results for community name
# Output files
csv_filepath = outputfolders['top']+"/"+output_filename+'.csv'
common_directory = outputfolders['top']+"/../"+output_filename

savefile = os.path.join(os.getcwd(), csv_filepath)
hui_incore_df_fixed.to_csv(savefile, index=False)
# Save second set of files in common directory
hui_incore_df_fixed.to_csv(common_directory+'.csv', index=False)

# After this they generate figures and create a PDF which we don't care about

/var/folders/dz/m2tmcg0161zf9_571f2hwzjh0000gr/T/ipykernel_32058/3755067937.py:6: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  hui_incore_df_fixed = hui_incore_df.applymap(lambda cell: int(cell) if str(cell).endswith('.0') else cell)


In [69]:
print("Not Using IN-CORE. Returning Dataframe")
hui_dataset_id = hui_incore_df_fixed

'OutputData/Brazos_TX_NSI/../hui_v2-0-0_48041_2010_rs9876'

#### Load Building Inventory

In [19]:
# Extract community-specific configuration
bldg_inv_id = community_dict['building_inventory']['id']

# Load building inventory based on source configuration
bldg_inv_gdf, bldg_inv_id, bldg_uniqueid = \
    workflow.load_building_inventory(community_dict) 

Brazos County, TX : county FIPS Code 48041
Directory OutputData already exists.
Making new directory to save output: OutputData/00_SourceData
Making new directory to save output: OutputData/00_SourceData/nsi_sec_usace_army_mil
   Saving file OutputData/00_SourceData/nsi_sec_usace_army_mil\nsi_01av1_hua_48041.shp


#### Generate Address Point Inventory

In [24]:
community_dict['building_inventory']

{'use_incore': False,
 'id': 'NSI',
 'note': 'NSI Building inventory for Brazos County, TX',
 'archetype_var': 'occtype',
 'bldg_uniqueid': 'fd_id_bid',
 'residential_archetypes': {'RES1-1SNB': {'Description': 'Single Family Dwelling',
   'HU estimate': 1},
  'RES1-1SWB': {'Description': 'Single Family Dwelling', 'HU estimate': 1},
  'RES1-2SNB': {'Description': 'Single Family Dwelling', 'HU estimate': 1},
  'RES1-2SWB': {'Description': 'Single Family Dwelling', 'HU estimate': 1},
  'RES1-3SNB': {'Description': 'Single Family Dwelling', 'HU estimate': 1},
  'RES1-3SWB': {'Description': 'Single Family Dwelling', 'HU estimate': 1},
  'RES1-SLNB': {'Description': 'Single Family Dwelling', 'HU estimate': 1},
  'RES1-SLWB': {'Description': 'Single Family Dwelling', 'HU estimate': 1},
  'RES1': {'Description': 'Single Family Dwelling', 'HU estimate': 1},
  'RES2': {'Description': 'Mobile / Manufactured Home', 'HU estimate': 1},
  'RES3A': {'Description': 'Duplex', 'HU estimate': 2},
  'RES3B

In [27]:
print(f"Generate Address point inventory for: {community}")
print(f"Based on building inventory: {bldg_inv_id}")

# Generate Address Point Inventory
addpt_inv_df, addpt_dataset_id = \
    workflow.generate_and_process_addpt(community, 
                                    housing_unit_inv_df, 
                                    bldg_inv_gdf, 
                                    community_dict,
                                    workflow.communities)

Generate Address point inventory for: Brazos_TX_NSI
Based on building inventory: NSI
Generate Address point inventory for: Brazos_TX_NSI
Based on building inventory: NSI
***************
Address Point Inventory Workflow
***************

Generating Address Point Inventory v2.0.0 data for Brazos_TX_NSI 2010
***************
Obtaining Census Block, Place, and PUMA Data
***************

File already exists: /Users/adamzs/Repos/intersect-community-data/OutputData/tl_2010_Brazos_TX_NSI_tabblockplacepuma10EPSG4269.csv
Converting blk104269 to Geodataframe
***************
Predicting Housing Unit Estimates
***************

Housing Unit Estimate File already exists: /Users/adamzs/Repos/intersect-community-data/OutputData/huest_v2-0-0_Brazos_TX_NSI_2010_NSI.csv
Brazos County, TX : county FIPS Code 48041
File already exists on local drive: /Users/adamzs/Repos/intersect-community-data/OutputData/addpt_v2-0-0_Brazos_TX_NSI_2010_NSI.csv
The Address Point Inventory ID contains a pandas dataframe


In [29]:
# Generate and process housing unit allocation
hua_hui_df = workflow.generate_and_process_hua(community,
                                            community_dict,
                                            housing_unit_inv_df,
                                            bldg_inv_id,
                                            bldg_uniqueid,
                                            bldg_inv_gdf,
                                            addpt_inv_df,
                                            outputfolders
                                            )  

Housing Unit Allocation for: Brazos_TX_NSI
Based on building inventory: NSI
Running up Housing Unit Allocation for Brazos_TX_NSI

***************************************
    Run Housing Unit Allocation for Brazos_TX_NSI
***************************************


***************************************
    Merge housing unit and address point data with first 3 counters.
***************************************

Round 1
Performing random merge at geography level: Block
Attempt to merge hui on all common group vars.
Running random merge by ['Block2010', 'huicounter1', 'ownershp1']
    Setting up  primary data with primary key and flags
Adding Block2010 to column list
Geolevels available []
Geolvarids available ['Block2010']
Adding Block2010 expected length 15
Dataframe has Block 2010 for new geovar Block2010
Confirming Block2010 has expected length.
Checking primary key name huid
Primary key variable huid is unique.
Primary key huid has no missing values
Initializing primary flag set variab

/Users/adamzs/Repos/intersect-community-data/pyncoda/ncoda_07d_run_hua_workflow.py:173: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[1.5 1.5 1.5 ... 1.5 1.5 1.5]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  addptv2_df.loc[condition1 & condition2 & condition3,'ownershp1'] \


Generating random merge order by ['Block2010', 'huicounter3', 'ownershp1']
    Setting up  secondary data with primary key and flags
Adding Block2010 to column list
Geolevels available []
Geolvarids available ['Block2010']
Adding Block2010 expected length 15
Dataframe has Block 2010 for new geovar Block2010
Confirming Block2010 has expected length.
Checking primary key name addptid
Primary key variable addptid is unique.
Primary key addptid has no missing values
['addptid', 'Block2010', 'blockid', 'placeGEOID10', 'COUNTYFP10', 'BLOCKID10_str', 'residential', 'bldgobs', 'flag_ap', 'occtype', 'huicounter1', 'huicounter2', 'huicounter3', 'ownershp1', 'ownershp2', 'ownershp3', 'strctid', 'strctid_flagsetrm', 'strctid_Block2010_flagsetrm', 'addrptid', 'fd_id_bid', 'huestimate', 'huicounter_addpt', 'placeNAME10', 'x', 'y']
Observations without primary flag set 59822
Observations without geovar flag set 59822
After updated observations without geovar flag set 59822
Setting 59822 flags for str

## Run Person Record File

In [ ]:
version = '3.0.0'
version_text = 'v3-0-0'

# open, read, and execute python program with reusable commands
from pyncoda.ncoda_07e_generate_prec import generate_prec_functions

# Save Outputfolder - due to long folder name paths output saved to folder with shorter name
# files from this program will be saved with the program name - 
# this helps to follow the overall workflow
outputfolder = "OutputData"
# Make directory to save output
if not os.path.exists(outputfolder):
    os.mkdir(outputfolder)

# Set random seed for reproducibility
seed = 1000
basevintage = 2010

generate_prec_df = generate_prec_functions(
                    communities =   communities,
                    seed =          seed,
                    version =       version,
                    version_text=   version_text,
                    basevintage=    basevintage,
                    outputfolder=   outputfolder
                    )

prec_df = generate_prec_df.generate_prec_v300()

## Explore and Validate Housing Unit Allocation


### Look at population characteristics and compare to US Census

In [ ]:
where = communities[community_id]['community_name']
print(where, focalplace, countyname, countyfips)

In [ ]:
# add race ethnicity to data frame for better map legends
hua_hui_race_gdf = PopResultsTable.add_race_ethnicity_to_pop_df(hua_hui_gdf)
hua_hui_race_gdf = PopResultsTable.add_hhinc_df(hua_hui_gdf)

# add category for missing building id
bldg_uniqueid = communities[community_id]['building_inventory']['bldg_uniqueid']
# add category for missing building id
buildingdata_conditions = {'cat_var' : {'variable_label' : 'Building Data Availability',
                         'notes' : 'Does Housing Unit have building data?'},
              'condition_list' : {
                1 : {'condition': f"(df['{bldg_uniqueid}'] == 'missing building id')", 'value_label': "0 Missing Building Data"},
                2 : {'condition': f"(df['{bldg_uniqueid}'] != 'missing building id')", 'value_label': "1 Building Data Available"}}
            }
hua_hui_gdf = add_label_cat_conditions_df(hua_hui_gdf, conditions = buildingdata_conditions)

In [ ]:
hua_hui_gdf.head()

In [ ]:
# set dataframe for focal place
focalplace_hua_hui_gdf =  hua_hui_gdf.loc[hua_hui_gdf['placeNAME10'] == focalplace].copy(deep=True)

In [ ]:
PopResultsTable.pop_results_table(
                  input_df = hua_hui_gdf, 
                  who = "Total Population by Households", 
                  what = "by Race, Ethnicity",
                  where = countyname,
                  when = "2010",
                  row_index = "Race Ethnicity",
                  col_index = 'Tenure Status')

In [ ]:
PopResultsTable.pop_results_table(
                  input_df = hua_hui_gdf, 
                  who = "Total Population by Households", 
                  what = "by Race, Ethnicity",
                  where = countyname,
                  when = "2010",
                  row_index = "Race Ethnicity",
                  col_index = 'Hispanic')

In [ ]:
PopResultsTable.pop_results_table(
                  input_df = prec_df, 
                  who = "Total Population by Persons", 
                  what = "by Race, Ethnicity",
                  where = countyname,
                  when = "2010",
                  row_index = "Race",
                  col_index = 'Hispanic')

In [ ]:
PopResultsTable.pop_results_table(hua_hui_gdf, 
                  who = "Total Population by Households", 
                  what = "by Race, Ethnicity",
                  where = where,
                  when = "2010",
                  row_index = "Race Ethnicity",
                  col_index = 'Building Data Availability_str',
                  row_percent = '0 Missing Building Data')

In [ ]:
try:
    print("Attempting to generate the population results table...")
    table1 = PopResultsTable.pop_results_table(
        focalplace_hua_hui_gdf,
        who="Total Population by Households",
        what="by Tenure",
        where=focalplace,
        when="2010",
        row_index="Tenure Status",
        col_index="Building Data Availability_str",
        row_percent="0 Missing Building Data"
    )
    print("Population results table generated successfully.")
except Exception as e:
    table1 = "no table generated"
    print(f"No Missing Building Data: {e}")

table1

In [ ]:
PopResultsTable.pop_results_table(hua_hui_gdf, 
                   who = "Median Household Income", 
                  what = "by Race, Ethnicity",
                  where = where,
                  when = "2010",
                  row_index = "Race Ethnicity",
                  col_index = 'Tenure Status')

In [ ]:
PopResultsTable.pop_results_table(focalplace_hua_hui_gdf, 
                   who = "Median Household Income", 
                  what = "by Race, Ethnicity",
                  where = focalplace,
                  when = "2010",
                  row_index = "Race Ethnicity",
                  col_index = 'Tenure Status')

#### Validate the Housing Unit Allocation has worked
Notice that the population count totals for the community
should match (pretty closely) data collected for the 2010 Decennial Census.
This can be confirmed by going to data.census.gov

In [ ]:
print("Total Population by Race and Ethnicity:")
print(f"https://data.census.gov/cedsci/table?g=050XX00US{countyfips}&tid=DECENNIALSF12010.P5")

print("Median Income by Race and Ethnicity:")
print(f"All Households: https://data.census.gov/cedsci/table?g=050XX00US{countyfips}&tid=ACSDT5Y2012.B19013")
print(f"Black Households: https://data.census.gov/cedsci/table?g=050XX00US{countyfips}&tid=ACSDT5Y2012.B19013B")
print(f"White, not Hispanic Households: https://data.census.gov/cedsci/table?g=050XX00US{countyfips}&tid=ACSDT5Y2012.B19013H")
print(f"Hispanic Households: https://data.census.gov/cedsci/table?g=050XX00US{countyfips}&tid=ACSDT5Y2012.B19013I")

Differences in the housing unit allocation and the Census count may be due to differences between political boundaries and the building inventory. See Rosenheim et al 2019 for more details.

The housing unit allocation, plus the building results will become the input for the social science models such as the population dislocation model.

## Explore Maps

In [ ]:
#mapname = 'hhincdotmap'
mapname = 'hhracedotmap'
# Map column
#map_var = 'Household Income Group'
map_var = 'Race Ethnicity'
place = focalplace

condition1 = "(hua_hui_race_gdf.race >= 1)"
condition2 = f"(hua_hui_race_gdf.placeNAME10 == '{place}')"
conditions = f"{condition1} & {condition2}"

county_hua_gdf = hua_hui_race_gdf.loc[eval(condition1)].copy(deep=True)
county_hua_gdf = county_hua_gdf.to_crs(epsg=4326)
focal_place_hua_gdf = hua_hui_race_gdf.loc[eval(conditions)].copy(deep=True)
focal_place_hua_gdf = focal_place_hua_gdf.to_crs(epsg=4326)

In [ ]:
from pyncoda.ncoda_04b_foliummaps import *
# plot png file
from IPython.display import Image

bldg_inv_id = communities[community_id]['building_inventory']['id']
outputfolder = 'OutputData'
community = communities[community_id]['community_name']

county_map = plot_dotmap_map(gdf=county_hua_gdf,
                        mapname=mapname,
                        map_var=map_var,
                        bldg_inv_id=bldg_inv_id,
                        community=community_id,
                        place = community,
                        outputfolder=outputfolder,
                        condition_id = "1",
                        basemap_source = cx.providers.CartoDB.Positron)

In [ ]:
# get xlim and ylim for focal place
xlim = [focal_place_hua_gdf.total_bounds[0], focal_place_hua_gdf.total_bounds[2]]
ylim = [focal_place_hua_gdf.total_bounds[1], focal_place_hua_gdf.total_bounds[3]]
print(xlim, ylim)

In [ ]:

focal_place_map = plot_dotmap_map(gdf=county_hua_gdf,
                        mapname=mapname,
                        map_var=map_var,
                        bldg_inv_id=bldg_inv_id,
                        community=community_id,
                        place = focalplace,
                        outputfolder=outputfolder,
                        condition_id = "2",
                        basemap_source = cx.providers.CartoDB.Positron,
                        xlim = xlim,
                        ylim = ylim,
                        focal_gdf = focal_place_hua_gdf)

In [ ]:
Image(focal_place_map+'.png', width= 800, height=800)

### Explore by Income

In [ ]:
mapname = 'hhincdotmap'

# Map column
map_var = 'Household Income Group'

# Ensure 'hhinc' is numeric
hua_hui_race_gdf['hhinc'] = pd.to_numeric(hua_hui_race_gdf['hhinc'], errors='coerce')

condition1 = "(hua_hui_race_gdf.hhinc >= 1)"

county_hua_gdf = hua_hui_race_gdf.loc[eval(condition1)].copy(deep=True)
county_hua_gdf = county_hua_gdf.to_crs(epsg=4326)

In [ ]:
# get xlim and ylim for focal place
xlim = [focal_place_hua_gdf.total_bounds[0], focal_place_hua_gdf.total_bounds[2]]
ylim = [focal_place_hua_gdf.total_bounds[1], focal_place_hua_gdf.total_bounds[3]]

focal_place_map = plot_dotmap_map(gdf=county_hua_gdf,
                        mapname=mapname,
                        map_var=map_var,
                        bldg_inv_id=bldg_inv_id,
                        community=community_id,
                        place = focalplace,
                        outputfolder=outputfolder,
                        condition_id = "2",
                        basemap_source = cx.providers.CartoDB.Positron,
                        xlim = xlim,
                        ylim = ylim,
                        focal_gdf = focal_place_hua_gdf)

In [ ]:
Image(focal_place_map+'.png', width= 800, height=800)

## View Codebook
The Housing Unit Allocation methodology generates a codebook for the housing unit inventory.

Look in the OutputData folder to find the codebook.